# Notebook 10: Ultrasonic & Tracking Sensors

## ADAS Connection
Your robot now has eyes -- but eyes alone are not enough. Real autonomous vehicles
use multiple sensor types together. A camera tells you what something looks like.
An ultrasonic sensor tells you exactly how far away it is. Tracking sensors tell
you where the boundaries are.

In this notebook you will add two new sensing capabilities to your robot:
- **Ultrasonic** -- measure distance to obstacles, like a car's parking sensors
- **Tracking** -- detect boundaries on the ground, like lane departure warning

---

## Part 1: Ultrasonic Sensor

### How It Works
The ultrasonic sensor works exactly like a bat or a submarine sonar. It sends out
a pulse of sound, waits for the echo to bounce back, and measures how long it took.
Since sound travels at a known speed, the time tells you the distance.

```
Sensor ──── ping ────▶ obstacle
Sensor ◀──── echo ──── obstacle
distance = (travel time x speed of sound) / 2
```

The sensor has two pins:
- **Trig** -- you send a short pulse to trigger a measurement
- **Echo** -- the sensor holds this HIGH for exactly as long as the echo takes to return

| Distance | What it means |
|----------|---------------|
| > 50 cm  | Clear path |
| 20-50 cm | Obstacle ahead, slow down |
| < 20 cm  | Too close, stop |

---

## Setup -- Run Once

In [ ]:
import RPi.GPIO as GPIO
import time

# ── Pin definitions (BCM numbering) ──────────────────────────
# Ultrasonic
TRIG = 1    # send ping
ECHO = 0    # receive echo

# Tracking sensors (left to right)
TRACK_L1 = 3    # far left
TRACK_L2 = 5    # center left
TRACK_R1 = 4    # center right
TRACK_R2 = 18   # far right

# Motors
LEFT_GO    = 20
LEFT_BACK  = 21
LEFT_PWM   = 16
RIGHT_GO   = 19
RIGHT_BACK = 6
RIGHT_PWM  = 13

GPIO.setmode(GPIO.BCM)
GPIO.setwarnings(False)

# Ultrasonic setup
GPIO.setup(TRIG, GPIO.OUT)
GPIO.setup(ECHO, GPIO.IN)
GPIO.output(TRIG, GPIO.LOW)

# Tracking setup
for pin in [TRACK_L1, TRACK_L2, TRACK_R1, TRACK_R2]:
    GPIO.setup(pin, GPIO.IN)

# Motor setup
for pin in [LEFT_GO, LEFT_BACK, LEFT_PWM, RIGHT_GO, RIGHT_BACK, RIGHT_PWM]:
    GPIO.setup(pin, GPIO.OUT)

pwm_left  = GPIO.PWM(LEFT_PWM,  100)
pwm_right = GPIO.PWM(RIGHT_PWM, 100)
pwm_left.start(0)
pwm_right.start(0)

# ── Motor functions ───────────────────────────────────────────
def forward(speed=60):
    GPIO.output(LEFT_GO,    GPIO.HIGH)
    GPIO.output(LEFT_BACK,  GPIO.LOW)
    GPIO.output(RIGHT_GO,   GPIO.HIGH)
    GPIO.output(RIGHT_BACK, GPIO.LOW)
    pwm_left.ChangeDutyCycle(speed)
    pwm_right.ChangeDutyCycle(speed)

def stop():
    GPIO.output(LEFT_GO,    GPIO.LOW)
    GPIO.output(LEFT_BACK,  GPIO.LOW)
    GPIO.output(RIGHT_GO,   GPIO.LOW)
    GPIO.output(RIGHT_BACK, GPIO.LOW)
    pwm_left.ChangeDutyCycle(0)
    pwm_right.ChangeDutyCycle(0)

def turn_left(speed=60, duration=0.3):
    GPIO.output(LEFT_GO,    GPIO.LOW)
    GPIO.output(LEFT_BACK,  GPIO.HIGH)
    GPIO.output(RIGHT_GO,   GPIO.HIGH)
    GPIO.output(RIGHT_BACK, GPIO.LOW)
    pwm_left.ChangeDutyCycle(speed)
    pwm_right.ChangeDutyCycle(speed)
    time.sleep(duration)
    stop()

def turn_right(speed=60, duration=0.3):
    GPIO.output(LEFT_GO,    GPIO.HIGH)
    GPIO.output(LEFT_BACK,  GPIO.LOW)
    GPIO.output(RIGHT_GO,   GPIO.LOW)
    GPIO.output(RIGHT_BACK, GPIO.HIGH)
    pwm_left.ChangeDutyCycle(speed)
    pwm_right.ChangeDutyCycle(speed)
    time.sleep(duration)
    stop()

# ── Sensor functions ──────────────────────────────────────────
def get_distance():
    """Measure distance in cm using the ultrasonic sensor."""
    # send a 10 microsecond trigger pulse
    GPIO.output(TRIG, GPIO.HIGH)
    time.sleep(0.00001)
    GPIO.output(TRIG, GPIO.LOW)

    # wait for echo to start
    pulse_start = time.time()
    while GPIO.input(ECHO) == 0:
        pulse_start = time.time()

    # wait for echo to end
    pulse_end = time.time()
    while GPIO.input(ECHO) == 1:
        pulse_end = time.time()

    # distance = (time x speed of sound) / 2
    # speed of sound = 34300 cm/s
    duration = pulse_end - pulse_start
    distance = (duration * 34300) / 2
    return round(distance, 1)

def get_tracking():
    """Read all four tracking sensors. Returns (L1, L2, R1, R2).
    1 = sensor over dark surface (on line/boundary)
    0 = sensor over light surface (clear)
    """
    return (
        GPIO.input(TRACK_L1),
        GPIO.input(TRACK_L2),
        GPIO.input(TRACK_R1),
        GPIO.input(TRACK_R2)
    )

time.sleep(0.5)   # let sensors settle
print('Setup complete!')
print('Ultrasonic and tracking sensors ready.')

---

## Ultrasonic -- Live Distance Reading

Run this cell to see distance readings in real time. Move your hand toward and away
from the front of the robot and watch the values change.

Press the **Stop** button or run the stop cell to end the loop.

In [ ]:
print('Reading distance -- move your hand in front of the sensor.')
print('Run the stop cell to end.\n')

reading_running = True
for _ in range(30):   # 30 readings then auto-stop
    if not reading_running:
        break
    dist = get_distance()
    bar = '#' * int(dist / 5)   # visual bar
    print(f'Distance: {dist:5.1f} cm  |{bar}')
    time.sleep(0.2)

print('\nDone.')

---

## YOUR TURN -- Tweak Zone 1: Obstacle Avoidance

The robot will drive forward and stop when it detects an obstacle within the threshold
distance. Tune the values and observe how the robot behaves.

- **STOP_DISTANCE** -- how close is too close?
- **DRIVE_SPEED** -- how fast is the robot moving?

> **Think like an engineer:** Real parking sensors beep faster as you get closer before
> finally stopping. How would you add that behavior here?

In [ ]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
STOP_DISTANCE = 20    # cm -- stop if obstacle closer than this
DRIVE_SPEED   = 50    # motor speed (0-100)
# ═══════════════════════════════════════

print(f'Obstacle avoidance active -- stop distance: {STOP_DISTANCE} cm')
print('Place an obstacle in front of the robot.')
print('Run the stop cell to end.\n')

avoid_running = True
forward(DRIVE_SPEED)

while avoid_running:
    dist = get_distance()
    print(f'Distance: {dist:5.1f} cm', end='  ')

    if dist < STOP_DISTANCE:
        stop()
        print('-- OBSTACLE DETECTED -- stopped')
        avoid_running = False
    else:
        print('-- clear')

    time.sleep(0.1)

In [ ]:
# STOP CELL -- run this to stop the robot at any time
avoid_running = False
stop()
print('Stopped.')

---

## Part 2: Tracking Sensors

### How They Work
Your robot has four infrared tracking sensors on the underside -- two on the left
(L1, L2) and two on the right (R1, R2). Each sensor shines infrared light downward
and detects how much bounces back.

- **Dark surface** (tape, black line) absorbs light -- sensor reads **1**
- **Light surface** (floor, white paper) reflects light -- sensor reads **0**

```
       front of robot
    [L1]  [L2]  [R1]  [R2]
```

By reading which sensors are over dark vs light, the robot knows if it is drifting
left or right of a boundary.

| L1 | L2 | R1 | R2 | Meaning |
|----|----|----|----|---------|
| 0  | 0  | 0  | 0  | All clear -- on light surface |
| 1  | 0  | 0  | 0  | Drifting left -- far left sensor hit boundary |
| 0  | 0  | 0  | 1  | Drifting right -- far right sensor hit boundary |
| 1  | 1  | 0  | 0  | Hard left -- correct right |
| 0  | 0  | 1  | 1  | Hard right -- correct left |
| 1  | 1  | 1  | 1  | Fully on dark surface |

---

## Tracking -- Live Sensor Reading

Run this cell and slide the robot across different surfaces. Watch how the sensor
values change as sensors cross the boundary.

> **Try it:** Place a piece of dark tape on the floor and slowly roll the robot over it.

In [ ]:
print('Reading tracking sensors -- move the robot over different surfaces.')
print('Run the stop cell to end.\n')
print('  L1   L2   R1   R2   Status')
print('  --   --   --   --   ------')

for _ in range(40):   # 40 readings then auto-stop
    l1, l2, r1, r2 = get_tracking()

    if   (l1, l2, r1, r2) == (0, 0, 0, 0):
        status = 'All clear'
    elif (l1, l2, r1, r2) == (1, 0, 0, 0):
        status = 'Drifting left'
    elif (l1, l2, r1, r2) == (0, 0, 0, 1):
        status = 'Drifting right'
    elif l1 == 1 and l2 == 1:
        status = 'Hard left'
    elif r1 == 1 and r2 == 1:
        status = 'Hard right'
    elif (l1, l2, r1, r2) == (1, 1, 1, 1):
        status = 'Fully on boundary'
    else:
        status = f'Mixed: {l1}{l2}{r1}{r2}'

    print(f'   {l1}    {l2}    {r1}    {r2}   {status}')
    time.sleep(0.25)

print('\nDone.')

---

## YOUR TURN -- Tweak Zone 2: Lane Assist

Place two parallel lines of dark tape on the floor to create a lane. The robot will
drive forward and correct itself if it drifts toward a boundary.

- **CORRECTION_DURATION** -- how long to turn when correcting
- **DRIVE_SPEED** -- how fast the robot moves

> **Think like an engineer:** Real lane assist does not jerk the wheel -- it makes small
> smooth corrections. How would you make your robot correct more gently?

In [ ]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
DRIVE_SPEED          = 50    # motor speed (0-100)
CORRECTION_DURATION  = 0.2   # seconds to turn when correcting -- try 0.1 to 0.4
# ═══════════════════════════════════════

print('Lane assist active.')
print('Place the robot between two dark tape lines.')
print('Run the stop cell to end.\n')

lane_running = True
forward(DRIVE_SPEED)

while lane_running:
    l1, l2, r1, r2 = get_tracking()

    if l1 == 1 or l2 == 1:
        # drifting left -- correct right
        print(f'  L1={l1} L2={l2} R1={r1} R2={r2}  --> drifting left, correcting right')
        turn_right(DRIVE_SPEED, CORRECTION_DURATION)
        forward(DRIVE_SPEED)

    elif r1 == 1 or r2 == 1:
        # drifting right -- correct left
        print(f'  L1={l1} L2={l2} R1={r1} R2={r2}  --> drifting right, correcting left')
        turn_left(DRIVE_SPEED, CORRECTION_DURATION)
        forward(DRIVE_SPEED)

    else:
        print(f'  L1={l1} L2={l2} R1={r1} R2={r2}  --> clear')

    time.sleep(0.1)

In [ ]:
# STOP CELL -- run this to stop the robot at any time
lane_running = False
stop()
print('Stopped.')

---

## What Happened?

Think about these questions with your team:

1. How close did the obstacle need to be before the robot stopped? Did the stop distance feel right?
2. Was the lane assist correction smooth or jerky? What tuning helped the most?
3. What would happen if you combined both sensors -- stop for obstacles AND stay in lane at the same time?
4. How would you use the AI model to make smarter decisions with these sensors?

---

## Always clean up when you are done!

In [ ]:
stop()
pwm_left.stop()
pwm_right.stop()
GPIO.cleanup()
print('GPIO cleaned up.')